# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashizhenya755-dev/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane: Refresh / Content Opportunity Scoring.

Task type: this is a scoring/ranking problem built on top of binary
classification. The underlying model predicts a probability (will this
page decline / does it need review), and that probability is then used to
rank pages into a review queue. It is not clustering (I'm not grouping
pages into unlabeled types) and it is not pure regression (I don't need an
exact numeric value, just a reliable ordering of "most worth reviewing
first" to "least").

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Proxy target (starter data, this week): trend_direction == "down" — a
binary label already present in the starter CSV, calculated from the
current 90-day window.

This is a proxy, not the real target, because it describes the current
state rather than a future outcome. The stronger version I'll move toward
later is a future-window label: features from a prior period predicting
decline or recovery in the NEXT period (e.g. prior 90 days -> next 30
days), once I work with the warehouse release. Using a same-window label
now is a known limitation, not something I'm claiming is ideal.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary metric: precision@K (specifically precision@50, matching the
starter pipeline's own evaluation).

I'm choosing precision@K over plain accuracy because the real decision
this supports is capacity-limited: a reviewer only has time to check a
fixed number of pages, so what matters is how many of the TOP-ranked
pages are actually worth reviewing — not how well the model scores every
page in the dataset. The starter pipeline's own results back this choice:
baseline rules score 0.240 precision@50, while a random forest reaches
0.740 precision@50 on the same data — roughly 3x more true positives in
the same top-50 review slots.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page, evaluated over its trailing 90-day window
(content_id is the unique key). Below, I load the starter dataset,
select the lane-relevant columns, and show what this looks like as an
actual dataframe.

In [15]:
!git clone https://github.com/yashizhenya755-dev/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [16]:
import subprocess
print(subprocess.run(['find', '/content', '-iname', '*.csv'], capture_output=True, text=True).stdout)

/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv



In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

# Columns relevant to Refresh / Content Opportunity Scoring
lane_cols = [
    "content_id", "client_id", "impressions_90d", "sessions_90d",
    "clicks_90d", "ctr", "avg_position", "content_age_days",
    "days_since_last_update", "word_count", "trend_direction"
]
lane_cols = [c for c in lane_cols if c in df.columns]

lane_df = df[lane_cols].copy()
print(f"Unit of analysis: one row = one content page (content_id)")
print(f"Shape: {lane_df.shape[0]:,} rows x {lane_df.shape[1]} columns")
lane_df.head(10)

Unit of analysis: one row = one content page (content_id)
Shape: 30,000 rows x 11 columns


,content_id,client_id,impressions_90d,sessions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,word_count,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,29,0.76,10.6,187,20,3221.0,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,7,0.05,20.3,445,25,2481.0,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,0.09,36.5,141,20,3515.0,down
3,content_331d6c4de07b,client_19581e27de,11751,78,58,0.49,6.2,463,22,NaN,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,24,0.13,44.0,263,14,2803.0,down
5,content_d4084a4bc775,client_f369cb89fc,3970,5,1,0.03,8.5,147,20,3080.0,down
6,content_9a34b442b552,client_8722616204,20,1,0,0.00,7.0,90,20,3059.0,down
7,content_a63219c6e95a,client_19581e27de,1724,28,1,0.06,21.2,445,22,NaN,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,68,29,0.09,46.0,90,20,3807.0,down
9,content_c27558df2b0c,client_19581e27de,1240,3,2,0.16,4.9,257,104,NaN,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (like "flag any page with a 90-day traffic drop over X%")
only ever looks at one or two signals at a time and can't weigh
interactions between them — e.g. a small drop combined with high
impressions and an aging page might matter more than a bigger drop on a
low-traffic page. The starter pipeline's own baseline-vs-model comparison
shows this isn't just theoretical: the hand-rule baseline gets 0.240
precision@50, while a random forest trained on the same observable
signals gets 0.740 precision@50 — evidence that a model combining many
weak signals ranks pages meaningfully better than a single fixed
threshold, on this starter slice.

This also matches the task type: ranking under limited review capacity is
exactly the kind of problem where relative ordering across many features
matters more than any single interpretable rule.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.